In [19]:
import torch 
import os
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
from torch.utils.data import * 
import numpy as np
torch.__version__

'2.12.0+cpu'

In [20]:
# 改进的配置
MAX_WORDS = 10000
MAX_LEN = 200
BATCH_SIZE = 64
EMB_SIZE = 128
HID_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.4
EPOCH_NUMBER = 10
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:
# 加载数据
x_train = np.load('./data/x_train.npy')
y_train = np.load('./data/y_train.npy')
x_val = np.load('./data/x_val.npy')
y_val = np.load('./data/y_val.npy')
print(x_train.shape, x_val.shape)

(31818, 200) (13636, 200)


In [22]:
# 转化为TensorDataset
train_data = TensorDataset(torch.LongTensor(x_train), torch.LongTensor(y_train))
val_data = TensorDataset(torch.LongTensor(x_val), torch.LongTensor(y_val))

In [23]:
# 转化为 DataLoader
train_sampler = RandomSampler(train_data)
train_loader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)
val_sampler = SequentialSampler(val_data)
val_loader = DataLoader(val_data, sampler=val_sampler, batch_size=BATCH_SIZE)

In [24]:
# 注意力机制
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)
        
    def forward(self, lstm_output):
        attn_weights = torch.softmax(self.attention(lstm_output), dim=1)
        weighted_output = lstm_output * attn_weights
        context = weighted_output.sum(dim=1)
        return context, attn_weights

# 改进的 BiLSTM + Attention 模型
class BiLSTMAttention(nn.Module):
    def __init__(self, max_words, emb_size, hid_size, num_layers, dropout):
        super(BiLSTMAttention, self).__init__()
        self.max_words = max_words
        self.emb_size = emb_size
        self.hid_size = hid_size
        self.num_layers = num_layers
        self.dropout = dropout
        
        self.Embedding = nn.Embedding(self.max_words, self.emb_size, padding_idx=0)
        
        # 双向 LSTM
        self.LSTM = nn.LSTM(
            self.emb_size, 
            self.hid_size, 
            num_layers=self.num_layers,
            dropout=self.dropout if self.num_layers > 1 else 0,
            batch_first=True,
            bidirectional=True
        )
        
        # 注意力层
        self.attention = Attention(self.hid_size * 2)
        
        self.dp = nn.Dropout(self.dropout)
        
        # 全连接层
        self.fc1 = nn.Linear(self.hid_size * 2, self.hid_size)
        self.fc2 = nn.Linear(self.hid_size, 2)
        
        # Layer Normalization
        self.layer_norm = nn.LayerNorm(self.hid_size * 2)
    
    def forward(self, x):
        x = self.Embedding(x)
        x = self.dp(x)
        
        lstm_out, _ = self.LSTM(x)
        lstm_out = self.layer_norm(lstm_out)
        
        context, _ = self.attention(lstm_out)
        context = self.dp(context)
        
        x = F.relu(self.fc1(context))
        x = self.dp(x)
        out = self.fc2(x)
        return out

In [25]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        y_ = model(x)
        loss = criterion(y_, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
        if (batch_idx + 1) % 50 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, (batch_idx + 1) * len(x), len(train_loader.dataset),
                100. * (batch_idx + 1) / len(train_loader), total_loss / (batch_idx + 1)))

In [26]:
def val(model, device, test_loader):
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='sum')
    val_loss = 0.0 
    acc = 0 
    for batch_idx, (x, y) in enumerate(test_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.no_grad():
            y_ = model(x)
        val_loss += criterion(y_, y)
        pred = y_.max(-1, keepdim=True)[1]
        acc += pred.eq(y.view_as(pred)).sum().item()
    val_loss /= len(test_loader.dataset)
    print('\\Val set: Average loss: {:.4f}, Accuracy: {}/{} ({:.1f}%)'.format(
        val_loss, acc, len(test_loader.dataset),
        100. * acc / len(test_loader.dataset)))
    return acc / len(test_loader.dataset)

In [27]:
# 创建改进模型
model = BiLSTMAttention(MAX_WORDS, EMB_SIZE, HID_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
print(model)

# 使用 Adam 优化器
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# 学习率调度
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_acc = 0.0 
PATH = 'imdb_model/model.pth'

for epoch in range(1, EPOCH_NUMBER + 1): 
    train(model, DEVICE, train_loader, optimizer, epoch)
    acc = val(model, DEVICE, val_loader)
    scheduler.step(acc)
    
    if best_acc < acc: 
        best_acc = acc 
        torch.save(model.state_dict(), PATH)
    print("acc is: {:.4f}, best acc is {:.4f}\n".format(acc, best_acc))

BiLSTMAttention(
  (Embedding): Embedding(10000, 128, padding_idx=0)
  (LSTM): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (attention): Attention(
    (attention): Linear(in_features=256, out_features=1, bias=True)
  )
  (dp): Dropout(p=0.4, inplace=False)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
)
Train Epoch: 1 [3200/31818 (10%)]	Loss: 0.695768
Train Epoch: 1 [6400/31818 (20%)]	Loss: 0.677013
Train Epoch: 1 [9600/31818 (30%)]	Loss: 0.651149
Train Epoch: 1 [12800/31818 (40%)]	Loss: 0.626900
Train Epoch: 1 [16000/31818 (50%)]	Loss: 0.604083
Train Epoch: 1 [19200/31818 (60%)]	Loss: 0.583473
Train Epoch: 1 [22400/31818 (70%)]	Loss: 0.570648
Train Epoch: 1 [25600/31818 (80%)]	Loss: 0.555390
Train Epoch: 1 [28800/31818 (90%)]	Loss: 0.549407
\Val set: Average loss: 0.4018, Accuracy: 11084/13636

In [28]:
# 检验保存的模型
best_model = BiLSTMAttention(MAX_WORDS, EMB_SIZE, HID_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
best_model.load_state_dict(torch.load(PATH))
val(best_model, DEVICE, val_loader)

\Val set: Average loss: 0.2947, Accuracy: 12101/13636 (88.7%)


0.887430331475506